In [3]:
import pandas as pd
import os

OLD_MODEL_INPUT = r'C:\projects\IdahoSTDM\ITD_STDM\Models\BaseYear\Development\IdahoSTDM\ITDSTDM\inputs'
OLD_MODEL_PATH = r'C:\projects\IdahoSTDM\ITD_STDM\Models\BaseYear\Development\IdahoSTDM\ITDSTDM\outputs_2010'
NEW_MODEL_INPUT = r'C:\projects\IdahoSTDM\ITD_STDM\Models\BaseYear\Development\ITDSTDM_06062025\inputs2020'
NEW_MODEL_PATH = r'C:\projects\IdahoSTDM\ITD_STDM\Models\BaseYear\Development\ITDSTDM_06062025\outputs2020'

old_taz = pd.read_csv(os.path.join(OLD_MODEL_INPUT, 'tazs.csv'))
new_taz = pd.read_csv(os.path.join(NEW_MODEL_INPUT, 'tazs.csv'))

census_acs = pd.read_excel(r'C:\projects\IdahoSTDM\ITD_STDM\Tasks\Task4_OnCallSupport\Census_data\acs_2023_emp_wfh.xlsx')

old_hh_data = pd.read_csv(os.path.join(OLD_MODEL_PATH, 'HouseholdData.csv')).merge(old_taz[['STDM_TAZ', 'State', 'County']], how = 'left', left_on = 'TAZ', right_on = 'STDM_TAZ')
new_hh_data = pd.read_csv(os.path.join(NEW_MODEL_PATH, 'HouseholdData.csv')).merge(new_taz[['STDM_TAZ', 'State', 'County']], how = 'left', left_on = 'TAZ', right_on = 'STDM_TAZ')

old_person_data = pd.read_csv(os.path.join(OLD_MODEL_PATH, 'PersonData.csv')).merge(old_taz[['STDM_TAZ', 'State', 'County']], how = 'left', left_on = 'home_taz', right_on = 'STDM_TAZ')
new_person_data = pd.read_csv(os.path.join(NEW_MODEL_PATH, 'PersonData.csv')).merge(new_taz[['STDM_TAZ', 'State', 'County']], how = 'left', left_on = 'home_taz', right_on = 'STDM_TAZ')

old_ldt_trips = pd.read_csv(os.path.join(OLD_MODEL_PATH, 'LDTPersonTrips.csv')).merge(old_person_data, how = 'left', left_on = ['hhID', 'memberID'], right_on = ['HH_ID', 'memberID']).merge(old_taz[['STDM_TAZ', 'State', 'County']].rename(columns = {'State': 'oState', 'County': 'oCounty'}), how = 'left', left_on = 'origin', right_on = 'STDM_TAZ').merge(old_taz[['STDM_TAZ', 'State', 'County']].rename(columns = {'State': 'dState', 'County': 'dCounty'}), how = 'left', left_on = 'destination', right_on = 'STDM_TAZ')
new_ldt_trips = pd.read_csv(os.path.join(NEW_MODEL_PATH, 'LDTPersonTrips.csv')).merge(new_person_data, how = 'left', left_on = ['hhID', 'memberID'], right_on = ['HH_ID', 'memberID']).merge(new_taz[['STDM_TAZ', 'State', 'County']].rename(columns = {'State': 'oState', 'County': 'oCounty'}), how = 'left', left_on = 'origin', right_on = 'STDM_TAZ').merge(new_taz[['STDM_TAZ', 'State', 'County']].rename(columns = {'State': 'dState', 'County': 'dCounty'}), how = 'left', left_on = 'destination', right_on = 'STDM_TAZ')

old_sdt_trips = pd.read_csv(os.path.join(OLD_MODEL_PATH, 'SDTPersonTrips.csv')).merge(old_person_data, how = 'left', left_on = ['hhID', 'memberID'], right_on = ['HH_ID', 'memberID']).merge(old_taz[['STDM_TAZ', 'State', 'County']].rename(columns = {'State': 'oState', 'County': 'oCounty'}), how = 'left', left_on = 'origin', right_on = 'STDM_TAZ').merge(old_taz[['STDM_TAZ', 'State', 'County']].rename(columns = {'State': 'dState', 'County': 'dCounty'}), how = 'left', left_on = 'destination', right_on = 'STDM_TAZ')
new_sdt_trips = pd.read_csv(os.path.join(NEW_MODEL_PATH, 'SDTPersonTrips.csv')).merge(new_person_data, how = 'left', left_on = ['hhID', 'memberID'], right_on = ['HH_ID', 'memberID']).merge(new_taz[['STDM_TAZ', 'State', 'County']].rename(columns = {'State': 'oState', 'County': 'oCounty'}), how = 'left', left_on = 'origin', right_on = 'STDM_TAZ').merge(new_taz[['STDM_TAZ', 'State', 'County']].rename(columns = {'State': 'dState', 'County': 'dCounty'}), how = 'left', left_on = 'destination', right_on = 'STDM_TAZ')


In [4]:
census_acs

,Unnamed: 0,NAME,County,tot_wrk,wfh,state,county,pct_wfh
0,0,"Ada County, Idaho",Ada,258161,43356,16,1,0.167942
1,1,"Adams County, Idaho",Adams,1723,264,16,3,0.153221
2,2,"Bannock County, Idaho",Bannock,39756,4166,16,5,0.104789
3,3,"Bear Lake County, Idaho",Bear Lake,2719,238,16,7,0.087532
4,4,"Benewah County, Idaho",Benewah,4001,280,16,9,0.069983
5,5,"Bingham County, Idaho",Bingham,21686,1287,16,11,0.059347
6,6,"Blaine County, Idaho",Blaine,13081,1743,16,13,0.133247
7,7,"Boise County, Idaho",Boise,3469,617,16,15,0.177861
8,8,"Bonner County, Idaho",Bonner,19860,2312,16,17,0.116415
9,9,"Bonneville County, Idaho",Bonneville,57168,5261,16,19,0.092027


# County-Level P & A

In [5]:
co_prod = old_hh_data[old_hh_data['State'] == 'Idaho'].groupby(['County']).agg(old_households = ('HH_ID', 'count')).join(
    new_hh_data[new_hh_data['State'] == 'Idaho'].groupby(['County']).agg(new_households = ('HH_ID', 'count'))
).fillna(0)
co_prod['hh_diff'] = co_prod['new_households'] - co_prod['old_households']
co_prod['hh_pct_diff'] = co_prod['hh_diff'] / co_prod['old_households']

co_prod = co_prod.join(
    old_sdt_trips.groupby(['oCounty']).agg(sdt_trips_old = ('origin', 'count'))
    ).join(
    new_sdt_trips.groupby(['oCounty']).agg(sdt_trips_new = ('origin', 'count'))
).fillna(0)
co_prod['sdt_diff'] = co_prod['sdt_trips_new'] - co_prod['sdt_trips_old']
co_prod['sdt_pct_diff'] = co_prod['sdt_diff'] / co_prod['sdt_trips_old']
co_prod = co_prod.join(
    old_ldt_trips.groupby(['oCounty']).agg(ldt_trips_old = ('origin', 'count'))
).join(
    new_ldt_trips.groupby(['oCounty']).agg(ldt_trips_new = ('origin', 'count'))
).fillna(0)
co_prod['ldt_diff'] = co_prod['ldt_trips_new'] - co_prod['ldt_trips_old']
co_prod['ldt_pct_diff'] = co_prod['ldt_diff'] / co_prod['ldt_trips_old']


co_prod['old_tot_prod'] = co_prod['sdt_trips_old'] + co_prod['ldt_trips_old'].fillna(0)
co_prod['new_tot_prod'] = co_prod['sdt_trips_new'] + co_prod['ldt_trips_new'].fillna(0)
co_prod['tot_diff'] = co_prod['new_tot_prod'] - co_prod['old_tot_prod']
co_prod['tot_pct_diff'] = co_prod['tot_diff'] / co_prod['old_tot_prod']
# # 

co_attr = old_hh_data[old_hh_data['State'] == 'Idaho'].groupby(['County']).agg(old_households = ('HH_ID', 'count')).join(
    new_hh_data.groupby(['County']).agg(new_households = ('HH_ID', 'count'))
).fillna(0)
co_attr['hh_diff'] = co_attr['new_households'] - co_attr['old_households']
co_attr['hh_pct_diff'] = co_attr['hh_diff'] / co_attr['old_households']

co_attr = co_attr.join(
    old_taz.groupby('County').agg(old_emp = ('TotEmp', 'sum'))).join(
        new_taz.groupby('County').agg(new_emp = ('TotEmp', 'sum'))
    )
co_attr['emp_diff'] = co_attr['new_emp'] - co_attr['old_emp']
co_attr['emp_pct_diff'] = co_attr['emp_diff'] / co_attr['old_emp']  

co_attr = co_attr.merge(census_acs[['County', 'tot_wrk', 'wfh']].rename(columns = {'tot_wrk': 'ACS Total Wrk', 'wfh': 'ACS WFH'}), how = 'left', on = 'County')
co_attr['ACS OOH Wrk'] = co_attr['ACS Total Wrk'] - co_attr['ACS WFH']


co_attr = co_attr.join(
    old_sdt_trips.groupby(['dCounty']).agg(sdt_trips_old = ('origin', 'count'))
    ).join(
    new_sdt_trips.groupby(['dCounty']).agg(sdt_trips_new = ('origin', 'count'))
).fillna(0)
co_attr['sdt_diff'] = co_attr['sdt_trips_new'] - co_attr['sdt_trips_old']
co_attr['sdt_pct_diff'] = co_attr['sdt_diff'] / co_attr['sdt_trips_old']
co_attr = co_attr.join(
    old_ldt_trips.groupby(['dCounty']).agg(ldt_trips_old = ('origin', 'count'))
).join(
    new_ldt_trips.groupby(['dCounty']).agg(ldt_trips_new = ('origin', 'count'))
).fillna(0)
co_attr['ldt_diff'] = co_attr['ldt_trips_new'] - co_attr['ldt_trips_old']
co_attr['ldt_pct_diff'] = co_attr['ldt_diff'] / co_attr['ldt_trips_old']


co_attr['old_tot_prod'] = co_attr['sdt_trips_old'] + co_attr['ldt_trips_old'].fillna(0)
co_attr['new_tot_prod'] = co_attr['sdt_trips_new'] + co_attr['ldt_trips_new'].fillna(0)
co_attr['tot_diff'] = co_attr['new_tot_prod'] - co_attr['old_tot_prod']
co_attr['tot_pct_diff'] = co_attr['tot_diff'] / co_attr['old_tot_prod']

# #
with pd.ExcelWriter('co_prod.xlsx') as writer:  
    co_prod.to_excel(writer, sheet_name='co_prod')
    co_attr.to_excel(writer, sheet_name='co_attr')


In [6]:

# sdt_diff = new_sdt_coflow - old_sdt_coflow
# sdt_pdiff = (sdt_diff / old_sdt_coflow).fillna(0).style.format("{:.1%}")

# old_ldt_co_p = old_ldt_trips.groupby(['oCounty']).agg(trips = ('origin', 'count')).reset_index().fillna(0)
# new_ldt_co_p = new_ldt_trips.groupby(['oCounty']).agg(trips = ('origin', 'count')).reset_index().fillna(0)

# ldt_diff = new_ldt_coflow - old_ldt_coflow
# ldt_pdiff = (ldt_diff / old_ldt_coflow).fillna(0).style.format("{:.1%}")

# old_total_coflow = old_sdt_coflow + old_ldt_coflow
# new_total_coflow = new_sdt_coflow + new_ldt_coflow
# tot_diff = new_total_coflow - old_total_coflow
# tot_pdiff = (tot_diff / old_total_coflow).fillna(0).style.format("{:.1%}")

# County Flows

In [7]:
old_sdt_coflow = old_sdt_trips.groupby(['oCounty', 'dCounty']).agg(trips = ('origin', 'count')).reset_index().pivot(index = 'oCounty', columns = 'dCounty', values = 'trips').fillna(0)
new_sdt_coflow = new_sdt_trips.groupby(['oCounty', 'dCounty']).agg(trips = ('origin', 'count')).reset_index().pivot(index = 'oCounty', columns = 'dCounty', values = 'trips').fillna(0)

sdt_diff = new_sdt_coflow - old_sdt_coflow
sdt_pdiff = (sdt_diff / old_sdt_coflow).fillna(0).style.format("{:.1%}")

old_ldt_coflow = old_ldt_trips.groupby(['oCounty', 'dCounty']).agg(trips = ('origin', 'count')).reset_index().pivot(index = 'oCounty', columns = 'dCounty', values = 'trips').fillna(0)
new_ldt_coflow = new_ldt_trips.groupby(['oCounty', 'dCounty']).agg(trips = ('origin', 'count')).reset_index().pivot(index = 'oCounty', columns = 'dCounty', values = 'trips').fillna(0)

ldt_diff = new_ldt_coflow - old_ldt_coflow
ldt_pdiff = (ldt_diff / old_ldt_coflow).fillna(0).style.format("{:.1%}")

old_total_coflow = old_sdt_coflow + old_ldt_coflow
new_total_coflow = new_sdt_coflow + new_ldt_coflow
tot_diff = new_total_coflow - old_total_coflow
tot_pdiff = (tot_diff / old_total_coflow).fillna(0).style.format("{:.1%}")

with pd.ExcelWriter('coflow_compare.xlsx') as writer:  
    old_sdt_coflow.to_excel(writer, sheet_name='old_sdt_coflow')
    new_sdt_coflow.to_excel(writer, sheet_name='new_sdt_coflow')
    sdt_diff.to_excel(writer, sheet_name='sdt_diff')
    sdt_pdiff.to_excel(writer, sheet_name='sdt_pdiff')

    old_ldt_coflow.to_excel(writer, sheet_name='old_ldt_coflow')
    new_ldt_coflow.to_excel(writer, sheet_name='new_ldt_coflow')
    ldt_diff.to_excel(writer, sheet_name='ldt_diff')
    ldt_pdiff.to_excel(writer, sheet_name='ldt_pdiff')

    old_total_coflow.to_excel(writer, sheet_name='old_total_coflow')
    new_total_coflow.to_excel(writer, sheet_name='new_total_coflow')
    tot_diff.to_excel(writer, sheet_name='tot_diff')
    tot_pdiff.to_excel(writer, sheet_name='tot_pdiff')